In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point

from data.constants import (WORLD_CRS, LOCAL_CRS,
                                OHARE_CENTROID, MIDWAY_CENTROID)
from data.geo import meter_to_foot, dms_to_decimal, foot_to_meter, foot_to_mi, meter_to_mi

In [2]:
train_file_in = "../data/raw/train_stations.geojson"
bus_stops_file_in = "../data/raw/bus_stops.geojson"
tract_file_in = "../data/raw/tracts.geojson"
poi_file_in = "../data/raw/poi_buildings.geojson"

# The merge between bike stations and bike rides was really complicated (5.x notebooks)
# Now we operate on the bike_rides data because it has correct imputed geoms.
bike_rides_in = "../data/interim/bike_rides_v2.geoparquet"

train_stations_out = "../data/interim/train_stations.geojson"
tract_file_out = "../data/interim/tracts.geoparquet"
bike_rides_out = "../data/interim/bike_rides_v3.geoparquet"

In [3]:
TOLERANCE = 200  # meters

# Pipeline in

In [4]:
train_stations = gpd.read_file(train_file_in).to_crs(LOCAL_CRS)
bus_stops = gpd.read_file(bus_stops_file_in).to_crs(LOCAL_CRS)
tract_points = gpd.read_file(tract_file_in).to_crs(LOCAL_CRS)
poi_buildings = gpd.read_file(poi_file_in).to_crs(LOCAL_CRS)
bike_rides = gpd.read_parquet(bike_rides_in).to_crs(LOCAL_CRS)

In [5]:
start_train_cols = train_stations.columns
start_tract_cols = tract_points.columns
start_bike_cols = bike_rides.columns

# Spatial Features

## Loop

In [6]:
loop = gpd.GeoSeries([Point(-87.62762846111136, 41.882370315275814)], crs=WORLD_CRS).to_crs(LOCAL_CRS).iloc[0]
train_stations['distance_loop'] = train_stations.geometry.distance(loop).apply(foot_to_meter)
tract_points['distance_loop'] = tract_points.geometry.distance(loop).apply(foot_to_meter)
bike_rides['distance_loop'] = bike_rides.geometry.distance(loop).apply(foot_to_meter)

## Airports

### -> CTA

We can label up the train and bus stops via name.

In [7]:
ohare_point = train_stations[train_stations.station_name == "O'Hare"].iloc[0].geometry
midway_point = train_stations[train_stations.station_name == "Midway"].iloc[0].geometry
train_stations['distance_ohare'] = train_stations.geometry.distance(ohare_point).apply(foot_to_meter)
train_stations['distance_midway'] = train_stations.geometry.distance(midway_point).apply(foot_to_meter)

### -> Uber

The rideshare pickups are anonymized to census area so we can't use building catchements.

In [8]:
oh_xy = dms_to_decimal(*OHARE_CENTROID[1]), dms_to_decimal(*OHARE_CENTROID[0])
oh_xy = Point(*oh_xy) # lng/lat
oh_xy = gpd.GeoSeries([oh_xy], crs=WORLD_CRS).to_crs(LOCAL_CRS).iloc[0]
oh_tract = tract_points.set_index('geoid10').geometry.to_crs(LOCAL_CRS).contains(oh_xy)

In [9]:
mdw_xy = dms_to_decimal(*MIDWAY_CENTROID[1]), dms_to_decimal(*MIDWAY_CENTROID[0])
mdw_xy = Point(*mdw_xy) # lng/lat
mdw_xy = gpd.GeoSeries([mdw_xy], crs=WORLD_CRS).to_crs(LOCAL_CRS).iloc[0]
mdw_tract = tract_points.set_index('geoid10').geometry.to_crs(LOCAL_CRS).contains(mdw_xy)

In [10]:
tract_points['distance_ohare'] = tract_points.geometry.distance(oh_xy).apply(foot_to_meter)
tract_points['distance_midway'] = tract_points.geometry.distance(mdw_xy).apply(foot_to_meter)

### -> Divvy

In [11]:
bike_rides['distance_ohare'] = bike_rides.geometry.distance(oh_xy).apply(foot_to_meter)
bike_rides['distance_midway'] = bike_rides.geometry.distance(mdw_xy).apply(foot_to_meter)

## POI catchements

In [12]:
def catchements(gdf: gpd.GeoDataFrame, pois:gpd.GeoSeries, areas=False):
    for row in pois.itertuples():
        name = row.name.replace(' ', '_')
        gdf[f"distance_{name}"] = gdf.geometry.distance(row.geometry).apply(foot_to_meter)
        # Measure overlapped differences.
        buffer = row.geometry.buffer(meter_to_foot(1600))
        if areas:
            gdf[f"contained_by_{name}"] = gdf.geometry.intersection(buffer).area / gdf.geometry.area
        else:
            gdf[f"contained_by_{name}"] = gdf.geometry.intersects(buffer).astype(float)
    return gdf

In [13]:
train_stations = train_stations.pipe(catchements, poi_buildings)
bike_rides = bike_rides.pipe(catchements, poi_buildings)
tract_points = tract_points.pipe(catchements, poi_buildings, areas=True)

## Tract Membership

Label the station tracts so we can aggregate to tract-level to match uber data.

In [14]:
def code_tract(gdf):
    """
    Spatial join point to census tract
    """
    coded_tracts = gdf[['geometry']].sjoin(tract_points[['geoid10','geometry']], how='left', predicate='within')
    gdf = pd.concat([gdf,
                    coded_tracts['geoid10'].rename('tract')],
                    axis=1)
    return gdf

In [15]:
train_stations = train_stations.pipe(code_tract)
bike_rides = bike_rides.pipe(code_tract)

In [16]:
print("Bike rides with unknown tract {:.2%}".format(
    bike_rides.tract.isna().mean()
))

Bike rides with unknown tract 0.26%


At this point 0.2% of bike rides are not strictly within a tract. 
But they happen to be really close, so let's

In [17]:
def code_nearest_tract(gdf):
    """
    Spatial join point to census tract
    """
    opts = dict(how='left', distance_col='dist', max_distance=meter_to_foot(TOLERANCE))
    coded_tracts = gdf[['geometry']].sjoin_nearest(tract_points[['geoid10','geometry']], **opts)
    assert all(coded_tracts.dist <= TOLERANCE)
    gdf = pd.concat([gdf,
                    coded_tracts['geoid10'].rename('tract')],
                    axis=1)
    return gdf

In [18]:
bike_rides_nearest = bike_rides[bike_rides.tract.isna()].drop(columns=['tract'])
bike_rides_nearest = code_nearest_tract(bike_rides_nearest)
# Just take first in case any are equidistant.
bike_rides_nearest = bike_rides_nearest.loc[bike_rides_nearest.index.drop_duplicates()]
assert bike_rides_nearest[['tract']].notna().all().all()

In [19]:
bike_rides = pd.concat([bike_rides[bike_rides.tract.notna()], bike_rides_nearest])

In [20]:
assert bike_rides.tract.notna().all()
assert train_stations.tract.notna().all()
assert tract_points.geoid10.notna().all()

## Voronoi Area

In [21]:
train_stations['voronoi'] = train_stations.voronoi_polygons().clip(tract_points.envelope)
train_stations['landarea'] = train_stations['voronoi'].area

tract_points = tract_points.assign(landarea = tract_points.area)

bike_ride_stations = bike_rides.geometry.drop_duplicates()
bike_voronoi = bike_ride_stations.voronoi_polygons().clip(tract_points.envelope)
bike_voronoi_map = dict(zip(bike_ride_stations, bike_voronoi))
bike_rides['voronoi'] = bike_rides.geometry.map(bike_voronoi_map)
bike_rides['landarea'] = bike_rides['voronoi'].area

train_stations['landarea'] = train_stations['landarea'].apply(np.sqrt).apply(foot_to_meter)
tract_points['landarea'] = tract_points['landarea'].apply(np.sqrt).apply(foot_to_meter)
bike_rides['landarea'] = bike_rides['landarea'].apply(np.sqrt).apply(foot_to_meter)

## Transit access distance

In [22]:
def access_stats(area_df, geoms, transit_name, area_id, voronoi=False):
    dist_col = f'{transit_name}_distance'
    area_centroids = (area_df
                      .assign(centroid = area_df.geometry.centroid)
                      .set_geometry('centroid')
                      .drop(columns='geometry')
                      .filter([area_id,'centroid']))
   
    # Note sjoin_nearest is m:m if points are equidistant ==> we further consolidate
    access = area_centroids.sjoin_nearest(geoms[['geometry']],
                                           how='left',
                                           exclusive=True,
                                           distance_col=dist_col) \
                        .groupby(area_id,as_index=False)[dist_col].min()
    access[dist_col] = access[dist_col].apply(foot_to_meter)

    geom_col = 'voronoi' if voronoi else 'geometry'
    area_geoms = area_df[[area_id, geom_col]].set_geometry(geom_col).set_crs(LOCAL_CRS)
    contained = area_geoms.sjoin(geoms[['geometry']], how='left', predicate='contains') \
                        .groupby(area_id,as_index=False)['index_right'].nunique() \
                        .rename(columns={'index_right':f'{transit_name}_contained'})
    
    area_df = area_df.merge(access).merge(contained)
    return area_df

In [23]:
tract_points = tract_points.pipe(access_stats, train_stations, 'train', 'geoid10')
tract_points = tract_points.pipe(access_stats, bike_rides, 'bike', 'geoid10')
tract_points = tract_points.pipe(access_stats, bus_stops, 'bus', 'geoid10')

train_stations = train_stations.pipe(access_stats, train_stations, 'train', 'map_id', True)
train_stations = train_stations.pipe(access_stats, bike_rides, 'bike', 'map_id', True)
train_stations = train_stations.pipe(access_stats, bus_stops, 'bus', 'map_id', True)

bike_rides = bike_rides.pipe(access_stats, train_stations, 'train', 'station_name', True)
bike_rides = bike_rides.pipe(access_stats, bike_rides, 'bike', 'station_name', True)
bike_rides = bike_rides.pipe(access_stats, bus_stops, 'bus', 'station_name', True)

## Aggregate demographics

In [24]:
def demogs(area_df, tracts, area_id, voronoi=False):
    geom_col = 'voronoi' if voronoi else 'geometry'
    area_geoms = area_df[[area_id, geom_col]].set_geometry(geom_col).set_crs(LOCAL_CRS)
    contained = area_geoms.sjoin(tracts, how='left', predicate='intersects') \
                        .drop(columns='index_right') \
                        .groupby(area_id,as_index=False) \
                        .agg({'TOTAL_POPULATION':'median', 'MEDIAN_HOUSEHOLD_INCOME':'median'})
    area_df = area_df.merge(contained)
    return area_df

In [25]:
train_stations = train_stations.pipe(demogs, tract_points, 'map_id', True)
bike_rides = bike_rides.pipe(demogs, tract_points, 'station_name', True)

## Long lat

In [26]:
def label_latlong(df):
    df['lat'] = df.geometry.centroid.y.apply(foot_to_meter)
    df['long'] = df.geometry.centroid.x.apply(foot_to_meter)
    return df

train_stations = train_stations.pipe(label_latlong)
bike_rides = bike_rides.pipe(label_latlong)
tract_points = tract_points.pipe(label_latlong)

# Scale distances

In [27]:
# Let's standardize lat/long because they are measured in millions of feet
# So doing so helps keep the relative scale of X vs Y in check. 
def zscale(x: pd.Series) -> pd.Series:
    return (x - x.mean()) / x.std()

def scale_cols(x: pd.DataFrame, columns: list[str] = None, like = None, scaler=zscale) -> pd.DataFrame:
    if columns is None:
        columns = []
    if like is not None:
        columns += list(x.filter(like=like).columns)
    return x.assign(**{col: scaler(x[col]) for col in columns})


In [28]:
train_stations = train_stations.pipe(scale_cols, ['lat','long','landarea'], scaler=meter_to_mi)
bike_rides = bike_rides.pipe(scale_cols, ['lat','long','landarea'], scaler=meter_to_mi)
tract_points = tract_points.pipe(scale_cols, ['lat','long','landarea'], scaler=meter_to_mi)

## Log Distances

In [29]:
# Add log distance.
for col in train_stations.columns:
    if 'distance' in col:
        train_stations['log_' + col] = train_stations[col].apply(np.log1p)
for col in bike_rides.columns:
    if 'distance' in col:
        bike_rides['log_' + col] = bike_rides[col].apply(np.log1p)
for col in tract_points.columns:
    if 'distance' in col:
        tract_points['log_' + col] = tract_points[col].apply(np.log1p)
train_stations['log_landarea'] = train_stations['landarea'].apply(np.log1p)
bike_rides['log_landarea'] = bike_rides['landarea'].apply(np.log1p)
tract_points['log_landarea'] = tract_points['landarea'].apply(np.log1p)

# Pipeline out

In [30]:
train_stations.drop(columns='voronoi').to_crs(WORLD_CRS).to_file(train_stations_out, index=False)
bike_rides.drop(columns='voronoi').to_crs(WORLD_CRS).to_parquet(bike_rides_out, index=False)
tract_points.to_crs(WORLD_CRS).to_parquet(tract_file_out, index=False)